# Projeto Aplicado III — Prova de Conceito do Sistema de Recomendação

Este notebook apresenta, passo a passo, a prova de conceito de um sistema de recomendação de receitas.

A ideia é deixar visível cada etapa pedida na Etapa 2 do projeto:

1. definição das bibliotecas Python;
2. carregamento dos dados;
3. análise exploratória;
4. tratamento e preparação da base;
5. treinamento de um modelo inicial baseado em conteúdo;
6. geração de recomendações;
7. avaliação preliminar do modelo.

O código foi escrito de forma simples, com nomes em português e sem uso de annotations, para facilitar a apresentação e explicação do grupo.


## Passo 1 — Importar as bibliotecas Python

Nesta etapa são importadas as bibliotecas utilizadas para leitura dos dados, análise exploratória, tratamento, vetorização textual, treinamento do modelo e avaliação.

In [ ]:
import ast
import math
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print('Bibliotecas importadas com sucesso.')

## Passo 2 — Definir os caminhos dos arquivos e parâmetros da prova de conceito

A base original é grande. Para esta prova de conceito, usamos uma amostra para o notebook rodar mais rápido. Na etapa final do projeto, o grupo pode aumentar a amostra ou usar a base completa.

In [2]:
pasta_dados = Path('/mnt/data')

arquivo_receitas = pasta_dados / 'RAW_recipes.csv'
arquivo_interacoes = pasta_dados / 'RAW_interactions.csv'
arquivo_treino = pasta_dados / 'interactions_train.csv'
arquivo_validacao = pasta_dados / 'interactions_validation.csv'
arquivo_teste = pasta_dados / 'interactions_test.csv'

usar_amostra = True
qtd_receitas = 30000 if usar_amostra else None
qtd_interacoes = 100000 if usar_amostra else None
qtd_treino = 100000 if usar_amostra else None
qtd_validacao = 5000 if usar_amostra else None
qtd_teste = 5000 if usar_amostra else None

qtd_receitas_modelo = 3000
qtd_recomendacoes = 10
nota_relevante = 4

print('Arquivos definidos:')
print(arquivo_receitas)
print(arquivo_interacoes)
print(arquivo_treino)
print(arquivo_validacao)
print(arquivo_teste)

Arquivos definidos:
/mnt/data/RAW_recipes.csv
/mnt/data/RAW_interactions.csv
/mnt/data/interactions_train.csv
/mnt/data/interactions_validation.csv
/mnt/data/interactions_test.csv


## Passo 3 — Carregar os dados

Aqui são carregadas as tabelas principais da base Food.com: receitas, interações brutas, treino, validação e teste.

In [3]:
receitas = pd.read_csv(
    arquivo_receitas,
    usecols=['id', 'name', 'minutes', 'tags', 'ingredients', 'n_ingredients', 'description'],
    nrows=qtd_receitas
)

interacoes = pd.read_csv(
    arquivo_interacoes,
    usecols=['user_id', 'recipe_id', 'rating', 'date'],
    nrows=qtd_interacoes
)

treino = pd.read_csv(arquivo_treino, nrows=qtd_treino)
validacao = pd.read_csv(arquivo_validacao, nrows=qtd_validacao)
teste = pd.read_csv(arquivo_teste, nrows=qtd_teste)

print('Receitas carregadas:', len(receitas))
print('Interações carregadas:', len(interacoes))
print('Registros de treino:', len(treino))
print('Registros de validação:', len(validacao))
print('Registros de teste:', len(teste))

Receitas carregadas: 30000
Interações carregadas: 100000
Registros de treino: 100000
Registros de validação: 5000
Registros de teste: 5000


## Passo 4 — Visualizar as primeiras linhas das receitas

Esta visualização ajuda a entender quais informações existem para cada receita, como nome, tempo de preparo, tags e ingredientes.

In [4]:
receitas.head()

,name,id,minutes,tags,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,"['60-minutes-or-less', 'time-to-make', 'course...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,"['30-minutes-or-less', 'time-to-make', 'course...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,"['time-to-make', 'course', 'preparation', 'mai...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,"['60-minutes-or-less', 'time-to-make', 'course...","this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,"['weeknight', 'time-to-make', 'course', 'main-...",my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


## Passo 5 — Visualizar as primeiras linhas das interações

As interações representam a relação entre usuários e receitas, principalmente por meio das avaliações dadas pelos usuários.

In [5]:
interacoes.head()

,user_id,recipe_id,date,rating
0,38094,40893,2003-02-17,4
1,1293707,40893,2011-12-21,5
2,8937,44394,2002-12-01,4
3,126440,85009,2010-02-27,5
4,57222,85009,2011-10-01,5


## Passo 6 — Verificar dimensões e colunas das bases

Nesta etapa verificamos o tamanho de cada DataFrame e quais colunas estão disponíveis.

In [6]:
print('Dimensão da base de receitas:', receitas.shape)
print('Colunas de receitas:', list(receitas.columns))

print()
print('Dimensão da base de interações:', interacoes.shape)
print('Colunas de interações:', list(interacoes.columns))

print()
print('Dimensão da base de treino:', treino.shape)
print('Colunas de treino:', list(treino.columns))

Dimensão da base de receitas: (30000, 7)
Colunas de receitas: ['name', 'id', 'minutes', 'tags', 'description', 'ingredients', 'n_ingredients']

Dimensão da base de interações: (100000, 4)
Colunas de interações: ['user_id', 'recipe_id', 'date', 'rating']

Dimensão da base de treino: (100000, 6)
Colunas de treino: ['user_id', 'recipe_id', 'date', 'rating', 'u', 'i']


## Passo 7 — Análise exploratória: quantidade de receitas, usuários e interações

Aqui verificamos quantas receitas foram carregadas, quantos usuários aparecem na amostra e quantas avaliações estão disponíveis.

In [7]:
qtd_usuarios = interacoes['user_id'].nunique()
qtd_receitas_avaliadas = interacoes['recipe_id'].nunique()
qtd_total_interacoes = len(interacoes)

print('Quantidade de receitas na base carregada:', receitas['id'].nunique())
print('Quantidade de usuários:', qtd_usuarios)
print('Quantidade de interações:', qtd_total_interacoes)
print('Quantidade de receitas avaliadas:', qtd_receitas_avaliadas)

Quantidade de receitas na base carregada: 30000
Quantidade de usuários: 38255
Quantidade de interações: 100000
Quantidade de receitas avaliadas: 19555


## Passo 8 — Análise exploratória: distribuição das notas

Esta etapa permite observar se as avaliações estão concentradas em notas altas ou baixas.

In [8]:
distribuicao_notas = interacoes['rating'].value_counts().sort_index().reset_index()
distribuicao_notas.columns = ['nota', 'quantidade']
distribuicao_notas['percentual'] = (distribuicao_notas['quantidade'] / distribuicao_notas['quantidade'].sum() * 100).round(2)

distribuicao_notas

,nota,quantidade,percentual
0,0,5181,5.18
1,1,1129,1.13
2,2,1281,1.28
3,3,3638,3.64
4,4,16679,16.68
5,5,72092,72.09


## Passo 9 — Análise exploratória: estatísticas das notas

Além da distribuição, calculamos média, mediana, mínimo e máximo das notas.

In [9]:
print('Média das notas:', round(interacoes['rating'].mean(), 4))
print('Mediana das notas:', interacoes['rating'].median())
print('Menor nota:', interacoes['rating'].min())
print('Maior nota:', interacoes['rating'].max())

Média das notas: 4.4178
Mediana das notas: 5.0
Menor nota: 0
Maior nota: 5


## Passo 10 — Análise exploratória: valores ausentes

Nesta etapa são identificados valores ausentes nas colunas da base de receitas e interações.

In [10]:
print('Valores ausentes nas receitas:')
display(receitas.isna().sum().sort_values(ascending=False))

print()
print('Valores ausentes nas interações:')
display(interacoes.isna().sum().sort_values(ascending=False))

Valores ausentes nas receitas:


description      720
name               1
id                 0
minutes            0
tags               0
ingredients        0
n_ingredients      0
dtype: int64


Valores ausentes nas interações:


user_id      0
recipe_id    0
date         0
rating       0
dtype: int64

## Passo 11 — Análise exploratória: tempo de preparo das receitas

O tempo de preparo pode ter valores extremos. Por isso, avaliamos estatísticas descritivas e percentis.

In [11]:
estatisticas_tempo = receitas['minutes'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
estatisticas_tempo

count     30000.000000
mean        127.988433
std        1984.322631
min           0.000000
25%          20.000000
50%          40.000000
75%          70.000000
90%         130.000000
95%         240.250000
99%        1225.250000
max      201610.000000
Name: minutes, dtype: float64

## Passo 12 — Análise exploratória: esparsidade da matriz usuário-receita

Em sistemas de recomendação, a matriz usuário-item costuma ser esparsa, pois cada usuário avalia apenas uma pequena parte dos itens disponíveis.

In [12]:
esparsidade = 1 - (qtd_total_interacoes / (qtd_usuarios * qtd_receitas_avaliadas))

print('Esparsidade da matriz usuário-receita:', round(esparsidade, 6))
print('Percentual de esparsidade:', round(esparsidade * 100, 4), '%')

Esparsidade da matriz usuário-receita: 0.999866
Percentual de esparsidade: 99.9866 %


## Passo 13 — Criar função simples para transformar textos de listas

Na base, ingredientes e tags aparecem como texto no formato de lista. Esta função transforma esse texto em uma lista real do Python.

In [13]:
def transformar_texto_em_lista(valor):
    if pd.isna(valor):
        return []

    try:
        lista = ast.literal_eval(valor)
        if isinstance(lista, list):
            return [str(item).lower().strip() for item in lista]
    except:
        return []

    return []

print('Função criada.')

Função criada.


## Passo 14 — Selecionar receitas mais frequentes no treino

Para a prova de conceito, selecionamos receitas que aparecem mais vezes nas interações de treino. Isso reduz o custo computacional e evita trabalhar com receitas muito raras no primeiro teste.

In [14]:
receitas_populares = (
    treino.groupby('recipe_id')['rating']
    .count()
    .sort_values(ascending=False)
    .head(qtd_receitas_modelo)
    .index
)

receitas_modelo = receitas[receitas['id'].isin(receitas_populares)].copy()

if len(receitas_modelo) < 500:
    receitas_modelo = receitas.copy()

print('Receitas selecionadas inicialmente:', len(receitas_modelo))

Receitas selecionadas inicialmente: 30000


## Passo 15 — Tratar a base de receitas

Nesta etapa são removidas duplicações, registros sem informações críticas e tempos inválidos. Também é criado um tempo tratado, limitando valores muito extremos pelo percentil 99.

In [15]:
receitas_modelo = receitas_modelo.drop_duplicates(subset='id')
receitas_modelo = receitas_modelo.dropna(subset=['id', 'name'])
receitas_modelo = receitas_modelo[receitas_modelo['minutes'].fillna(0) > 0]

limite_tempo = receitas_modelo['minutes'].quantile(0.99)
receitas_modelo['tempo_tratado'] = receitas_modelo['minutes'].clip(upper=limite_tempo)

print('Receitas após tratamento:', len(receitas_modelo))
print('Limite aplicado ao tempo de preparo:', limite_tempo)

Receitas após tratamento: 29856
Limite aplicado ao tempo de preparo: 1254.5000000000073


## Passo 16 — Preparar ingredientes, tags e texto final da receita

Para o modelo baseado em conteúdo, juntamos nome, ingredientes e tags em um único campo de texto chamado `texto_receita`. Esse texto será transformado em números pelo TF-IDF.

In [16]:
receitas_modelo['ingredientes_lista'] = receitas_modelo['ingredients'].apply(transformar_texto_em_lista)
receitas_modelo['tags_lista'] = receitas_modelo['tags'].apply(transformar_texto_em_lista)

receitas_modelo['texto_receita'] = (
    receitas_modelo['name'].fillna('') + ' ' +
    receitas_modelo['ingredientes_lista'].apply(lambda lista: ' '.join(lista)) + ' ' +
    receitas_modelo['tags_lista'].apply(lambda lista: ' '.join(lista))
)

receitas_modelo = receitas_modelo[['id', 'name', 'minutes', 'tempo_tratado', 'n_ingredients', 'texto_receita']].reset_index(drop=True)

print('Base preparada para o modelo:', receitas_modelo.shape)
receitas_modelo.head()

Base preparada para o modelo: (29856, 6)


,id,name,minutes,tempo_tratado,n_ingredients,texto_receita
0,137739,arriba baked winter squash mexican style,55,55.0,7,arriba baked winter squash mexican style win...
1,31490,a bit different breakfast pizza,30,30.0,6,a bit different breakfast pizza prepared pizz...
2,112140,all in the kitchen chili,130,130.0,13,all in the kitchen chili ground beef yellow o...
3,59389,alouette potatoes,45,45.0,11,alouette potatoes spreadable cheese with garl...
4,44061,amish tomato ketchup for canning,190,190.0,8,amish tomato ketchup for canning tomato juic...


## Passo 17 — Treinar o modelo baseado em conteúdo com TF-IDF

O TF-IDF transforma o texto das receitas em uma matriz numérica. Cada receita passa a ser representada por um vetor de características textuais.

In [17]:
vetorizador = TfidfVectorizer(
    stop_words='english',
    min_df=2,
    max_df=0.90,
    max_features=15000
)

matriz_receitas = vetorizador.fit_transform(receitas_modelo['texto_receita'])

print('Modelo TF-IDF treinado.')
print('Formato da matriz TF-IDF:', matriz_receitas.shape)

Modelo TF-IDF treinado.
Formato da matriz TF-IDF: (29856, 4690)


## Passo 18 — Treinar o KNN para buscar receitas parecidas

Nesta prova de conceito, o KNN é usado para encontrar receitas vizinhas, ou seja, receitas mais parecidas com a receita escolhida. A métrica usada é a distância do cosseno.

In [18]:
modelo_vizinhos = NearestNeighbors(metric='cosine', algorithm='brute')
modelo_vizinhos.fit(matriz_receitas)

print('Modelo KNN treinado para recomendação baseada em conteúdo.')

Modelo KNN treinado para recomendação baseada em conteúdo.


## Passo 19 — Criar função para recomendar receitas parecidas

A função recebe uma receita base e retorna outras receitas semelhantes com base no conteúdo textual de ingredientes e tags.

In [19]:
def recomendar_receitas_parecidas(id_receita, quantidade=5):
    posicoes = pd.Series(receitas_modelo.index, index=receitas_modelo['id']).to_dict()

    if id_receita not in posicoes:
        print('Receita não encontrada no conjunto usado pela prova de conceito.')
        return pd.DataFrame()

    posicao_receita = posicoes[id_receita]
    distancias, indices = modelo_vizinhos.kneighbors(
        matriz_receitas[posicao_receita],
        n_neighbors=quantidade + 1
    )

    recomendacoes = receitas_modelo.iloc[indices.flatten()].copy()
    recomendacoes['similaridade'] = 1 - distancias.flatten()
    recomendacoes = recomendacoes[recomendacoes['id'] != id_receita]

    return recomendacoes[['id', 'name', 'minutes', 'n_ingredients', 'similaridade']].head(quantidade)

print('Função de recomendação criada.')

Função de recomendação criada.


## Passo 20 — Gerar uma recomendação de exemplo

Agora escolhemos uma receita da base preparada e pedimos ao modelo para encontrar receitas parecidas.

In [20]:
id_exemplo = receitas_modelo.iloc[0]['id']
nome_exemplo = receitas_modelo.iloc[0]['name']

print('Receita base:')
print(id_exemplo, '-', nome_exemplo)

recomendacoes_exemplo = recomendar_receitas_parecidas(id_exemplo, quantidade=5)
recomendacoes_exemplo

Receita base:
137739 - arriba   baked winter squash mexican style


,id,name,minutes,n_ingredients,similaridade
15445,254360,baked winter squash,40,5,0.616577
15446,272858,baked winter squash au gratin,70,9,0.525654
15142,261048,baked spaghetti squash,50,4,0.480701
15447,141614,baked winter squash soup,165,13,0.460586
15259,240027,baked stuffed winter squash,95,10,0.455535


## Passo 21 — Avaliar uma linha de base com média por receita

Antes de avaliar modelos mais complexos, criamos uma linha de base simples. A previsão da nota é a média histórica da receita no treino. Quando a receita não aparece no treino, usamos a média geral.

In [21]:
media_geral = treino['rating'].mean()
media_por_receita = treino.groupby('recipe_id')['rating'].mean()

notas_reais = validacao['rating']
notas_previstas = validacao['recipe_id'].map(media_por_receita).fillna(media_geral)

rmse = math.sqrt(mean_squared_error(notas_reais, notas_previstas))
mae = mean_absolute_error(notas_reais, notas_previstas)

print('RMSE da linha de base:', round(rmse, 4))
print('MAE da linha de base:', round(mae, 4))
print('Média geral das notas no treino:', round(media_geral, 4))

RMSE da linha de base: 1.3115
MAE da linha de base: 0.8259
Média geral das notas no treino: 4.5989


## Passo 22 — Criar perfis simples de usuários

Para avaliar listas de recomendação, criamos um perfil textual para cada usuário com base nas receitas que ele avaliou bem no treino.

In [22]:
receita_para_posicao = pd.Series(receitas_modelo.index, index=receitas_modelo['id']).to_dict()

treino_filtrado = treino[treino['recipe_id'].isin(receita_para_posicao)].copy()
treino_bem_avaliado = treino_filtrado[treino_filtrado['rating'] >= nota_relevante]

perfis_usuarios = {}
receitas_vistas = {}

for usuario, grupo in treino_bem_avaliado.groupby('user_id'):
    posicoes = []

    for id_receita in grupo['recipe_id']:
        if id_receita in receita_para_posicao:
            posicoes.append(receita_para_posicao[id_receita])

    if len(posicoes) > 0:
        perfis_usuarios[usuario] = np.asarray(matriz_receitas[posicoes].mean(axis=0))
        receitas_vistas[usuario] = set(treino_filtrado[treino_filtrado['user_id'] == usuario]['recipe_id'])

print('Perfis de usuários criados:', len(perfis_usuarios))

Perfis de usuários criados: 3040


## Passo 23 — Criar função para recomendar receitas para um usuário

A recomendação para usuário compara o perfil do usuário com todas as receitas disponíveis e retorna as mais parecidas que ele ainda não viu.

In [23]:
def recomendar_para_usuario(usuario, quantidade=10):
    if usuario not in perfis_usuarios:
        return []

    pontuacoes = cosine_similarity(perfis_usuarios[usuario], matriz_receitas).flatten()
    ranking = np.argsort(pontuacoes)[::-1]
    ja_vistas = receitas_vistas.get(usuario, set())

    recomendacoes = []

    for posicao in ranking:
        id_receita = receitas_modelo.iloc[posicao]['id']
        if id_receita not in ja_vistas:
            recomendacoes.append(id_receita)
        if len(recomendacoes) == quantidade:
            break

    return recomendacoes

print('Função de recomendação por usuário criada.')

Função de recomendação por usuário criada.


## Passo 24 — Testar recomendação para um usuário

Aqui escolhemos um usuário com perfil criado e geramos uma lista de receitas recomendadas.

In [24]:
if len(perfis_usuarios) > 0:
    usuario_exemplo = list(perfis_usuarios.keys())[0]
    recomendadas_usuario = recomendar_para_usuario(usuario_exemplo, quantidade=10)

    print('Usuário exemplo:', usuario_exemplo)
    print('IDs recomendados:', recomendadas_usuario)

    display(receitas_modelo[receitas_modelo['id'].isin(recomendadas_usuario)][['id', 'name', 'minutes', 'n_ingredients']])
else:
    print('Não houve usuários suficientes para gerar recomendação nesta amostra.')

Usuário exemplo: 1533
IDs recomendados: [np.int64(15179), np.int64(12867), np.int64(83822), np.int64(356538), np.int64(45302), np.int64(12660), np.int64(447530), np.int64(56376), np.int64(246657), np.int64(22829)]


,id,name,minutes,n_ingredients
10734,22829,authentic italian spaghetti sauce,65,7
14174,12660,baked frittata for one,40,13
15351,56376,baked tomato,15,5
18689,356538,basil zucchini,10,5
24163,83822,black bean onion soup,30,9
29177,12867,broccoli cheese soup for the soul,30,9
29288,15179,broccoli pasta in a fresh tomato sauce,25,12
29393,447530,broccoli soup for dieters,55,10
29421,246657,broccoli with a garlic and lemon dressing,10,6
29454,45302,broccoli with garlic and parmesan cheese,20,6


## Passo 25 — Avaliar Precision@K e Recall@K

Nesta avaliação, uma receita é considerada relevante quando recebeu nota maior ou igual a 4 na validação. O objetivo é verificar se as receitas recomendadas aparecem entre as receitas relevantes do usuário.

In [25]:
receitas_disponiveis = set(receitas_modelo['id'])

validacao_relevante = validacao[
    (validacao['rating'] >= nota_relevante) &
    (validacao['recipe_id'].isin(receitas_disponiveis))
]

relevantes_por_usuario = validacao_relevante.groupby('user_id')['recipe_id'].apply(set).to_dict()

precisoes = []
revocacoes = []
usuarios_avaliados = list(relevantes_por_usuario.keys())[:200]

for usuario in usuarios_avaliados:
    if usuario not in perfis_usuarios:
        continue

    recomendadas = recomendar_para_usuario(usuario, quantidade=qtd_recomendacoes)
    relevantes = relevantes_por_usuario[usuario]
    acertos = len(set(recomendadas) & relevantes)

    precisoes.append(acertos / qtd_recomendacoes)
    revocacoes.append(acertos / len(relevantes))

if len(precisoes) == 0:
    print('Não houve usuários suficientes para calcular Precision@K e Recall@K nesta amostra.')
else:
    print('Precision@10:', round(np.mean(precisoes), 4))
    print('Recall@10:', round(np.mean(revocacoes), 4))
    print('Usuários avaliados:', len(precisoes))

Precision@10: 0.0
Recall@10: 0.0
Usuários avaliados: 118


## Passo 26 — Síntese da prova de conceito

Nesta etapa, a prova de conceito mostrou que é possível:

- carregar a base Food.com;
- analisar a estrutura dos dados;
- preparar textos de ingredientes e tags;
- criar vetores TF-IDF para representar receitas;
- usar KNN com similaridade do cosseno para recomendar receitas parecidas;
- criar uma linha de base para previsão de notas;
- iniciar a avaliação com RMSE, MAE, Precision@K e Recall@K.

Na próxima etapa do projeto, o grupo pode evoluir essa abordagem para um sistema híbrido, combinando recomendação baseada em conteúdo com filtragem colaborativa.